In [0]:
# 00_apply_migrations — sql/databricks/migrations/*.sql 을 순서대로 실행
# DDL 원본: sql/databricks/migrations/
# 이 노트북은 스키마/테이블 생성만 담당한다. 데이터 적재는 하지 않는다.

In [0]:
dbutils.widgets.text("catalog", "access_drift")
dbutils.widgets.text("skip_bronze_silver_ddl", "false")

CATALOG = dbutils.widgets.get("catalog")
SKIP_BRONZE_SILVER_DDL = dbutils.widgets.get("skip_bronze_silver_ddl").strip().lower() == "true"
MIGRATIONS_BASE = "/Workspace/Shared/access-drift/files/sql/databricks/migrations"

In [0]:
import os

if not os.path.isdir(MIGRATIONS_BASE):
    raise RuntimeError(
        f"Migration 파일 디렉터리를 찾을 수 없습니다: {MIGRATIONS_BASE}\n"
        "bundle deploy가 완료됐는지 확인하세요."
    )

In [0]:
# migration 파일별 변수 치환 정의
def strip_line_comments(sql: str) -> str:
    return "\n".join(
        line for line in sql.splitlines()
        if not line.strip().startswith("--")
    )

migrations = [
    ("001_create_bronze_tables.sql", {"catalog": CATALOG, "schema": "bronze"}),
    ("002_create_silver_tables.sql", {"catalog": CATALOG, "schema": "silver"}),
    ("003_create_eval_tables.sql",   {"catalog": CATALOG, "schema": "eval"}),
    ("004_create_intermediate_tables.sql", {"catalog": CATALOG, "schema": "intermediate"}),
    ("005_create_gold_tables.sql",         {"catalog": CATALOG, "schema": "gold"}),
    ("006_create_context_tables.sql",      {"catalog": CATALOG, "schema": "context"}),
    ("007_create_app_tables.sql",          {"catalog": CATALOG, "schema": "app"}),
    ("008_create_rag_tables.sql",          {"catalog": CATALOG, "schema": "rag"}),
    # RAG migrations are appended after the existing base RAG table migration.
    # They create/extend only DS-owned rag.* contracts and do not overwrite bronze/silver data.
    # Keeping them in this ordered list lets the standard migration job verify the RAG schema.
    ("009_create_rag_chain_tables.sql",    {"catalog": CATALOG, "schema": "rag"}),
    ("010_create_rag_operational_tables.sql", {"catalog": CATALOG, "schema": "rag"}),
]

if SKIP_BRONZE_SILVER_DDL:
    for schema in ["bronze", "silver"]:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    migrations = [
        item for item in migrations
        if item[0] not in {"001_create_bronze_tables.sql", "002_create_silver_tables.sql"}
    ]
    print("Bronze/Silver DDL skipped; DLT owns B/S dataset lifecycle")

for filename, variables in migrations:
    path = os.path.join(MIGRATIONS_BASE, filename)
    with open(path) as f:
        sql = f.read()

    for key, value in variables.items():
        sql = sql.replace(f"${{{key}}}", value)

    sql = strip_line_comments(sql)
    statements = [
        s.strip() for s in sql.split(";")
        if s.strip()
    ]

    for stmt in statements:
        spark.sql(stmt)

    print(f"{filename}: {len(statements)} statements applied")

001_create_bronze_tables.sql: 7 statements applied
002_create_silver_tables.sql: 5 statements applied
003_create_eval_tables.sql: 2 statements applied
004_create_intermediate_tables.sql: 1 statements applied
005_create_gold_tables.sql: 1 statements applied


In [0]:
# Contract evolution for existing legacy Delta tables.
# CREATE TABLE IF NOT EXISTS does not alter already-created tables.
if SKIP_BRONZE_SILVER_DDL:
    print("Bronze/Silver contract evolution skipped; DLT owns B/S dataset lifecycle")
else:
    bronze_identity_columns = {
        row.col_name
        for row in spark.sql(
            f"SHOW COLUMNS IN {CATALOG}.bronze.bronze_identity_lifecycle_seed"
        ).collect()
    }
    if "display_name" not in bronze_identity_columns:
        spark.sql(
            f"ALTER TABLE {CATALOG}.bronze.bronze_identity_lifecycle_seed "
            "ADD COLUMNS (display_name STRING)"
        )
        print("bronze_identity_lifecycle_seed.display_name column added")
    else:
        print("bronze_identity_lifecycle_seed.display_name column already exists")

    silver_edges_columns = {
        row.col_name
        for row in spark.sql(
            f"SHOW COLUMNS IN {CATALOG}.silver.silver_edges"
        ).collect()
    }
    if "last_used_at" not in silver_edges_columns:
        spark.sql(
            f"ALTER TABLE {CATALOG}.silver.silver_edges "
            "ADD COLUMNS (last_used_at TIMESTAMP)"
        )
        print("silver_edges.last_used_at column added")
    else:
        print("silver_edges.last_used_at column already exists")

    for table_name in (
        "bronze_github_access_edges",
        "bronze_google_workspace_access_edges",
        "bronze_slack_access_edges",
    ):
        columns = {
            row.col_name
            for row in spark.sql(f"SHOW COLUMNS IN {CATALOG}.bronze.{table_name}").collect()
        }
        if "last_used_at" not in columns:
            spark.sql(
                f"ALTER TABLE {CATALOG}.bronze.{table_name} "
                "ADD COLUMNS (last_used_at TIMESTAMP)"
            )
            print(f"{table_name}.last_used_at column added")
        else:
            print(f"{table_name}.last_used_at column already exists")

    # Backward-compatible schema guard for the RAG evidence fields used downstream.
    # These ALTER TABLE statements are idempotent: they only ADD missing columns.
    # Existing rows, bronze/silver ingestion outputs, and DE-owned table contents are not overwritten.
    # This prevents older dev workspaces from failing strict column checks in 05/06 after RAG fields are introduced.
    intermediate_columns = {
        row.col_name
        for row in spark.sql(
            f"SHOW COLUMNS IN {CATALOG}.intermediate.nhi_residual_access_findings"
        ).collect()
    }
    intermediate_add_columns = [
        ("principal_kind", "STRING"),
        ("source_system", "STRING"),
        ("principal_type", "STRING"),
        ("finding_type_label", "STRING"),
        ("risk_factors", "ARRAY<STRING>"),
    ]
    missing_intermediate_columns = [
        f"{name} {dtype}" for name, dtype in intermediate_add_columns
        if name not in intermediate_columns
    ]
    if missing_intermediate_columns:
        spark.sql(
            f"ALTER TABLE {CATALOG}.intermediate.nhi_residual_access_findings "
            f"ADD COLUMNS ({', '.join(missing_intermediate_columns)})"
        )
        print(f"nhi_residual_access_findings columns added: {missing_intermediate_columns}")
    else:
        print("nhi_residual_access_findings optional columns already exist")

    gold_columns = {
        row.col_name
        for row in spark.sql(f"SHOW COLUMNS IN {CATALOG}.gold.gold_core").collect()
    }
    gold_add_columns = [
        ("principal_kind", "STRING"),
        ("source_system", "STRING"),
        ("principal_type", "STRING"),
        ("finding_type_label", "STRING"),
        ("risk_factors", "ARRAY<STRING>"),
    ]
    missing_gold_columns = [
        f"{name} {dtype}" for name, dtype in gold_add_columns
        if name not in gold_columns
    ]
    if missing_gold_columns:
        spark.sql(
            f"ALTER TABLE {CATALOG}.gold.gold_core "
            f"ADD COLUMNS ({', '.join(missing_gold_columns)})"
        )
        print(f"gold_core columns added: {missing_gold_columns}")
    else:
        print("gold_core optional columns already exist")


## Smoke test

In [0]:
expected_tables = {
    "bronze": [
        "bronze_azure_applications",
        "bronze_azure_application_owners",
        "bronze_azure_password_credentials",
        "bronze_identity_lifecycle_seed",
        "bronze_assets_seed",
        "bronze_access_edges_seed",
    ],
    "silver": [
        "silver_principals",
        "silver_credentials",
        "silver_assets",
        "silver_edges",
    ],
    "eval": [
        "ground_truth_case",
    ],
    "intermediate": [
        "nhi_residual_access_findings",
    ],
    "gold": [
        "gold_core",
    ],
    "context": [
        "asset_context",
        "assignment_routing",
    ],
    "app": [
        "risk_action_state",
        "risk_action_event",
    ],
    "rag": [
        # DS-owned RAG contract tables. These checks prove table availability only;
        # they do not execute RAG generation or mutate upstream DE datasets.
        "doc_chunks",
        "recommended_action",
        "eval_results",
        "source_docs_manifest",
        "remediation_history",
        "governance_assessment",
        "cost_estimate",
    ],
}

if SKIP_BRONZE_SILVER_DDL:
    expected_tables.pop("bronze", None)
    expected_tables.pop("silver", None)

for schema, tables in expected_tables.items():
    existing = {
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").collect()
    }
    missing = sorted(set(tables) - existing)
    assert not missing, f"Missing tables in {CATALOG}.{schema}: {missing}"

print("migration smoke test passed")
print("migrations complete")

migration smoke test passed
migrations complete
